# Plot multi-lead predictions for predicting hurricane track (distance) errors.
author: Elizabeth A. Barnes and Randal J. Barnes

In [9]:
%matplotlib inline
%load_ext autotime

import sys
import os
import importlib as imp
import warnings
from shapely.errors import ShapelyDeprecationWarning

warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy as ct
import plots
import compute_predictions

import experiment_settings
import mahalanobis

mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams["figure.dpi"] = 100
dpiFig = 600
plots.set_plot_rc()
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 1.41 ms (started: 2023-11-27 12:34:48 -07:00)


In [22]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "16 December 2022"

array(['testcase', 'production_run_OFDXY', 'production_run_OBDXY',
       'test_run_OBDXY', 'test_run_OFDXY', 'test_2023season_OFDV',
       'test_2023season_OBDV', 'forecast_test'], dtype='<U20')

time: 4.71 ms (started: 2023-11-20 13:06:10 -07:00)


In [2]:
testing = experiment_settings.Experiments()
print('index range and short names for experiments')
for expname in testing.get_exp_list_short:
    exp_inds = [index for index, exp_string in enumerate(testing.get_exp_list) if expname in exp_string]
    print(np.min(exp_inds), '-', np.max(exp_inds)+1, expname)

index range and short names for experiments
0 - 20 dev_OFD
20 - 40 dev_OBD
time: 2.28 ms (started: 2023-11-27 12:31:16 -07:00)


In [3]:
# pick the experiment
expname = 'dev_OFD'

# get the correct indices for these experiments
exp_inds = [index for index, exp_string in enumerate(testing.get_exp_list) if expname in exp_string]

time: 424 µs (started: 2023-11-27 12:31:23 -07:00)


In [4]:
EXP_NAME_VEC = testing.get_exp_list[np.min(exp_inds):np.max(exp_inds)+1]
EXP_NAME_VEC

['dev_OFD_AL12',
 'dev_OFD_AL24',
 'dev_OFD_AL36',
 'dev_OFD_AL48',
 'dev_OFD_AL60',
 'dev_OFD_AL72',
 'dev_OFD_AL84',
 'dev_OFD_AL96',
 'dev_OFD_AL108',
 'dev_OFD_AL120',
 'dev_OFD_EP12',
 'dev_OFD_EP24',
 'dev_OFD_EP36',
 'dev_OFD_EP48',
 'dev_OFD_EP60',
 'dev_OFD_EP72',
 'dev_OFD_EP84',
 'dev_OFD_EP96',
 'dev_OFD_EP108',
 'dev_OFD_EP120']

time: 1.61 ms (started: 2023-11-27 12:31:31 -07:00)


In [7]:
DATA_PATH = "data/"
MODEL_PATH = os.path.join("saved_models", expname)
FIGURE_PATH = os.path.join("figures/analysis/", expname)
PREDICTIONS_PATH = os.path.join("saved_predictions/", expname)

if not os.path.exists(FIGURE_PATH):
    os.makedirs(FIGURE_PATH)

time: 950 µs (started: 2023-11-27 12:31:54 -07:00)


# Plot Results

In [8]:
storm_dict = {
    "IAN": {"storm_name": "IAN",
            "year": 2022,
            "extent": [-100,-65,5,35],
            "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
            # https://www.nhc.noaa.gov/aboutcone.shtml
            },
    "FIONA": {"storm_name": "FIONA",
              "year": 2022,
              "extent": [-100,-45,10,55],
              "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
              },
    "IRMA": {"storm_name": "IRMA",
             "year": 2017,
             "extent": [-100,-20,10,40],
             "nhc_cone_radius": {0:8, 12:29, 24:45, 36:63, 48:78, 60:107, 72:107, 96:159, 120:211}
             # https://www.air-worldwide.com/blog/posts/2017/8/the-ever-shrinking-cone-of-uncertainty/
             },
    "NICOLE": {"storm_name": "NICOLE",
               "year": 2022,
               "extent": [-100,-48,20,45],
               "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
               },
    "JULIA": {"storm_name": "JULIA",
              "year": 2022,
              "extent": [-100,-65,5,20],
              "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
              },
    "NORMAN": {"storm_name": "NORMAN",
                "year": 2018,
                "pred_time": 90306,
                "extent": [195, 360-135, 5, 35],
                "nhc_cone_radius": {0:8, 12:25, 24:40, 36:51, 48:66, 60:93, 72:93, 96:116, 120:151}
                },
    "HARVEY": {"storm_name": "HARVEY",
                "year": 2017,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:29, 24:45, 36:63, 48:78, 60:107, 72:107, 96:159, 120:211}
                },
    "DORIAN": {"storm_name": "DORIAN",
                "year": 2019,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:26, 24:41, 36:54, 48:68, 60:102, 72:102, 96:151, 120:198}
                },
    "OTIS": {"storm_name": "OTIS",
                "year": 2023,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200},
                "basin": "EP"
                },
    "PHILIPPE": {"storm_name": "PHILIPPE",
                "year": 2023,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200},
                "basin": "AL"
                },
}

time: 1.3 ms (started: 2023-11-27 12:31:58 -07:00)


In [13]:
imp.reload(mahalanobis)
imp.reload(plots)
imp.reload(compute_predictions)
import glob
KM_TO_DEG = 1.0 / 111.

for storm_name in ("PHILIPPE",):#("IAN","IRMA","FIONA"):#("IAN", "NICOLE", "IRMA"):
    print(storm_name)
    storm = storm_dict[storm_name]

    for RNG_SEED in testing.get_experiment(EXP_NAME_VEC[0])['rng_seed_list']:

        TESTING_YEAR = storm["year"]
        files = glob.glob(PREDICTIONS_PATH + '/*')
        
        # GET PREDICTIONS
        df_pred_test = pd.DataFrame()
        for file in files:
            try:
                df = pd.read_csv(file)
            except:
                continue

            rng_seed = RNG_SEED
            years_test = (TESTING_YEAR,)
            df["exp_name"] = expname
            df_pred_test = pd.concat([df_pred_test, df], axis=0)

        #------------------------------------------------------------
        # MAKE THE PLOTS
        df = df_pred_test.loc[
            (df_pred_test["NAME"] == storm["storm_name"]) & (df_pred_test["YEAR"] == TESTING_YEAR)].copy()
        df = df.sort_values("MMDDHH").reset_index(drop=True)
        forecast_dates = df["MMDDHH"].unique()

        for i,pred_time in enumerate(forecast_dates):
            print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
            storm["pred_time"] = pred_time
            df_storm = df_pred_test.loc[
                (df_pred_test["NAME"] == storm["storm_name"]) & (df_pred_test["MMDDHH"] == storm["pred_time"])].copy()
            df_storm = compute_predictions.add_lead_zero(df_storm)
            # df_storm["nhc_cone_radius"] = [storm["nhc_cone_radius"][key] for key in df_storm["FHOUR"].unique()]
            
            # get dynamic extent
            extx = [df_storm["LONN"], df_storm["LONN"] + KM_TO_DEG * df_storm["OFDX"]]
            exty = [df_storm["LATN"], df_storm["LATN"] + KM_TO_DEG * df_storm["OFDY"]]
            storm_extent = [-(360-np.min(extx)+5), -(360-np.max(extx)-5), np.min(exty)-5, np.max(exty)+5]
            
            # plot probability ellipses
            fig = plt.figure(dpi=150, )
            ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            details = plots.plot_probability_ellipses(
                df_storm,
                ax=ax,
                leadtimes=np.arange(0,120+12, 12),
                contours=(.1, .25, .5, .75, .9,),
                # extent = storm["extent"],
                extent = storm_extent,
                alpha=.4,
                vector=True,
                plot_nhc_cone=False,
            )
            plt.gca().get_legend().remove()
            # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
            ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
            plt.savefig(
                FIGURE_PATH + '/probability_ellipses_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                dpi=dpiFig,
                bbox_inches='tight',
            )
            plt.close()

            # plot banana cones
            try:
                fig = plt.figure(dpi=150, )
                ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
                details = plots.plot_banana_of_uncertainty(
                    df_storm=df_storm,
                    ax=ax,
                    # extent=storm["extent"],
                    extent=storm_extent,
                    vector=True,
                    colors=("steelblue","khaki"),
                    alpha=.75,
                    plot_nhc_cone=False,
                )
                # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
                ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
                plt.savefig(
                    FIGURE_PATH + '/banana_cone_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                    dpi=dpiFig,
                    bbox_inches='tight',
                )
                plt.close()
            except:
                print('not enough data for spline computation. not making the figure.')
                plt.close()

PHILIPPE
1 of 49: 92318
not enough data for spline computation. not making the figure.
2 of 49: 92400
not enough data for spline computation. not making the figure.
3 of 49: 92406
not enough data for spline computation. not making the figure.
4 of 49: 92412
not enough data for spline computation. not making the figure.
5 of 49: 92418
not enough data for spline computation. not making the figure.
6 of 49: 92500
not enough data for spline computation. not making the figure.
7 of 49: 92506
not enough data for spline computation. not making the figure.
8 of 49: 92512
not enough data for spline computation. not making the figure.
9 of 49: 92518
not enough data for spline computation. not making the figure.
10 of 49: 92600
not enough data for spline computation. not making the figure.
11 of 49: 92606
not enough data for spline computation. not making the figure.
12 of 49: 92612
not enough data for spline computation. not making the figure.
13 of 49: 92618
not enough data for spline computati